<a href="https://colab.research.google.com/github/tioluwaniiyin123/ELEN-5301/blob/Benchmarking/homework4%20torch.profiler_G_P_U.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch torchvision

In [2]:


# EOS PyTorch version
!wget https://raw.githubusercontent.com/dionhaefner/pyhpc-benchmarks/master/benchmarks/equation_of_state/eos_pytorch.py


--2025-10-20 03:21:48--  https://raw.githubusercontent.com/dionhaefner/pyhpc-benchmarks/master/benchmarks/equation_of_state/eos_pytorch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7318 (7.1K) [text/plain]
Saving to: ‘eos_pytorch.py’

eos_pytorch.py      100%[===================>]   7.15K  --.-KB/s    in 0s      

2025-10-20 03:21:48 (121 MB/s) - ‘eos_pytorch.py’ saved [7318/7318]



In [3]:
!python eos_pytorch.py

In [4]:
import torch
import torchvision.models as models
from torch.profiler import profile, ProfilerActivity, record_function
from eos_pytorch import gsw_dHdT

In [5]:
import torch
from torch.profiler import profile, ProfilerActivity, record_function
from torchvision import models
from eos_pytorch import gsw_dHdT

# Choose device (CPU or GPU if available)
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    print(" CUDA not available — running on CPU only.")

model = models.resnet18().to(device)
inputs = torch.randn(5, 3, 224, 224).to(device)

# Build activities list
activities = [ProfilerActivity.CPU]
if device == "cuda":
    activities += [ProfilerActivity.CUDA]

# Sort key for profiler output
sort_by_keyword = "cuda_memory_usage" if device == "cuda" else "cpu_memory_usage"

# Start profiling
with profile(
    activities=activities,

    record_shapes=True,
    profile_memory=True,
    with_stack=True,
    on_trace_ready=torch.profiler.tensorboard_trace_handler(f'./log/{device}')
) as prof:
    with record_function("model_inference"):
        output = model(inputs)
        # calling the function
        gsw_dHdT(inputs, torch.randn_like(inputs), torch.randn_like(inputs))
    prof.step()

# Print profiling summary
print(prof.key_averages().table(sort_by=sort_by_keyword, row_limit=10))




-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::mul         0.38%       4.690ms         2.68%      33.372ms     171.140us       1.183ms        32.73%       1.203ms       6.169us           0 B           0 B     146.43 MB     146.43 M